# Task 3 — Car Price Prediction with Machine Learning

CoreAxis Technology Data Science Internship.

This notebook covers data quality checks, feature engineering, exploratory analysis, regression modeling, hold-out evaluation, and five-fold cross-validation.

## 1. Load and Inspect Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('../data/car_data.csv')
print('Shape:', df.shape)
display(df.head())
print('Missing values:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())

## 2. Data Cleaning and Feature Engineering

In [ ]:
df = df.drop_duplicates().copy()
df['Car_Age'] = 2018 - df['Year']
print('Rows after duplicate removal:', len(df))
print(df.describe(include='all').transpose())

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['Selling_Price'], bins=30)
plt.title('Selling Price Distribution')
plt.xlabel('Selling Price (₹ lakh)')
plt.ylabel('Number of Vehicles')
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(df['Present_Price'], df['Selling_Price'], alpha=0.7)
plt.title('Selling Price vs Present Price')
plt.xlabel('Present Price (₹ lakh)')
plt.ylabel('Selling Price (₹ lakh)')
plt.show()

numeric_cols = ['Year','Selling_Price','Present_Price','Driven_kms','Owner','Car_Age']
corr = df[numeric_cols].corr()
display(corr)

## 4. Prepare Features and Preprocessing

In [ ]:
X = df.drop(columns='Selling_Price')
y = df['Selling_Price']
categorical_features = X.select_dtypes(include='object').columns.tolist()
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features)
], remainder='passthrough')

## 5. Train and Evaluate Regression Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=500, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
results = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append([name, mean_absolute_error(y_test,pred), mean_squared_error(y_test,pred)**0.5, r2_score(y_test,pred)])

results_df = pd.DataFrame(results, columns=['Model','MAE','RMSE','R2'])
display(results_df.sort_values('MAE'))

## 6. Five-Fold Cross-Validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring={
        'MAE':'neg_mean_absolute_error', 'RMSE':'neg_root_mean_squared_error', 'R2':'r2'
    }, n_jobs=-1)
    cv_rows.append({
        'Model':name,
        'Mean_MAE':-scores['test_MAE'].mean(),
        'Mean_RMSE':-scores['test_RMSE'].mean(),
        'Mean_R2':scores['test_R2'].mean()
    })
cv_results = pd.DataFrame(cv_rows).sort_values('Mean_RMSE')
display(cv_results)

## 7. Conclusion

Five-fold validation identifies Gradient Boosting as the strongest overall model among the three tested approaches. Present_Price is the strongest linear signal associated with Selling_Price, while Car_Age and categorical variables also provide predictive information. See `IMPORTANT_FINDINGS.md` and the exported CSV results for the final figures.